In [ ]:
# %%
!pip install -q transformers torch torchvision pandas numpy Pillow opencv-python-headless scipy matplotlib seaborn tqdm

# %%
import os
import time
import datetime
import warnings
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings('ignore')

In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Exact Google Drive Paths Requested
BASE_DRIVE_DIR = '/content/drive/My Drive/PRICES'
IMAGES_DIR = '/content/drive/My Drive/PRICES/Images'
DATA_DIR = '/content/drive/My Drive/PRICES/DATA'
FIGURES_DIR = os.path.join(DATA_DIR, 'CV_Figures')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Input & Output Files in /DATA
STRUCTURAL_CSV = os.path.join(DATA_DIR, 'property_listings_multicity_filtered_15km.csv')
SPATIAL_CSV = os.path.join(DATA_DIR, 'property_listings_multicity_spatial.csv')

IMAGE_LEVEL_CSV = os.path.join(DATA_DIR, 'cv_image_level_details.csv')
HEDONIC_CSV = os.path.join(DATA_DIR, 'cv_features_hedonic.csv')
MERGED_AUDIT_CSV = os.path.join(DATA_DIR, 'cv_structural_merged_audit.csv')
TABLE_SCENE_TEX = os.path.join(DATA_DIR, 'table_cv_scene_authenticity.tex')
TABLE_STATS_TEX = os.path.join(DATA_DIR, 'table_cv_summary_stats.tex')
TABLE_TESTS_TEX = os.path.join(DATA_DIR, 'table_cv_hypothesis_tests.tex')
LOG_TEX = os.path.join(DATA_DIR, 'log.tex')

BATCH_SIZE = 50

# 3. Hardware Accelerator Detection
device = 'cpu'
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
    except ImportError:
        device = torch.device('cpu')

print(f'[INFO] Running CLIP inference on device: {device}')
print(f'[INFO] Images directory exists: {os.path.exists(IMAGES_DIR)}')
print(f'[INFO] 15km Structural CSV exists: {os.path.exists(STRUCTURAL_CSV)}')

# %%
# 4. Load Pre-Trained CLIP Model & Semantic Taxonomies
MODEL_ID = 'openai/clip-vit-base-patch32'
model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

AUTH_PROMPTS = [
    'a real camera photograph of a real house or room',
    'a 3D architectural CGI render or digital computer illustration of a house',
    'a 2D architectural floorplan, blueprint, map, or site plan',
    'a real estate promotional flyer, logo, or advertising banner with text'
]
AUTH_LABELS = ['real_photo', 'cgi_render', 'floorplan_map', 'promo_flyer']

SCENE_PROMPTS = [
    'an exterior front facade of a house from the street',
    'an outdoor garden, backyard, patio, or green park area',
    'an indoor living room, dining room, or hallway',
    'an indoor kitchen with counters, sink, or cabinets',
    'an indoor bedroom or closet',
    'an indoor bathroom, toilet, or shower',
    'a gated community entrance, parking garage, or street view'
]
SCENE_LABELS = [
    'exterior_facade', 'outdoor_garden_patio', 'interior_living_dining',
    'interior_kitchen', 'interior_bedroom', 'interior_bathroom', 'street_parking_gate'
]

COND_PROMPTS = [
    'a poorly maintained house in bad condition, old, worn, and damaged',
    'a well-maintained house in excellent condition, clean, and newly painted'
]

GREEN_PROMPTS = [
    'an indoor room without outdoor nature',
    'an outdoor view of concrete, asphalt street, or barren dirt',
    'an outdoor view of a house with lush greenery, trees, grass, or a garden'
]

FINISH_PROMPTS = [
    'an outdoor building facade or street',
    'an outdated room with old, basic, or cheap finishes and materials',
    'a modern renovated room with high-end luxury finishes, granite, or modern lighting'
]

ALL_PROMPTS = AUTH_PROMPTS + SCENE_PROMPTS + COND_PROMPTS + GREEN_PROMPTS + FINISH_PROMPTS
text_inputs = processor(text=ALL_PROMPTS, return_tensors='pt', padding=True).to(device)

# %%
# 5. Batched Folder-Level Photometric & CLIP Inference Engine
def compute_photometrics(pil_img, file_path):
    w, h = pil_img.size
    file_kb = os.path.getsize(file_path) / 1024.0
    rgb = np.array(pil_img, dtype=np.float32)
    gray = cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY)

    brightness = float(np.mean(gray))
    contrast = float(np.std(gray))
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    R, G, B = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    colorfulness = float(np.sqrt(np.var(rg) + np.var(yb)) + 0.3 * np.sqrt(np.mean(rg)**2 + np.mean(yb)**2))

    return w, h, round(w / max(h, 1), 3), round(file_kb, 2), round(brightness, 2), round(contrast, 2), round(sharpness, 2), round(colorfulness, 2)

def analyze_folder_batch(folder_path, listing_id, img_files):
    pil_images = []
    meta_records = []

    for img_f in img_files:
        full_path = os.path.join(folder_path, img_f)
        try:
            pil_img = Image.open(full_path).convert('RGB')
            w, h, aspect, file_kb, bright, contrast, sharp, color = compute_photometrics(pil_img, full_path)
            pil_images.append(pil_img)
            meta_records.append({
                'listing_id': listing_id,
                'image_file': img_f,
                'width_px': w,
                'height_px': h,
                'megapixels': round((w * h) / 1e6, 4),
                'aspect_ratio': aspect,
                'file_size_kb': file_kb,
                'brightness': bright,
                'contrast': contrast,
                'sharpness': sharp,
                'colorfulness': color
            })
        except Exception:
            continue

    if not pil_images:
        return []

    img_inputs = processor(images=pil_images, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(
            input_ids=text_inputs['input_ids'],
            attention_mask=text_inputs['attention_mask'],
            pixel_values=img_inputs['pixel_values']
        )
        logits_batch = outputs.logits_per_image.detach().cpu().numpy()

    results = []
    for idx, logits in enumerate(logits_batch):
        def softmax_slice(start, end):
            sub = logits[start:end]
            e_x = np.exp(sub - np.max(sub))
            return e_x / e_x.sum()

        p_auth = softmax_slice(0, 4)
        auth_type = AUTH_LABELS[int(np.argmax(p_auth))]
        auth_entropy = float(-np.sum(p_auth * np.log(np.clip(p_auth, 1e-9, 1.0))))

        p_scene = softmax_slice(4, 11)
        scene_type = SCENE_LABELS[int(np.argmax(p_scene))]
        scene_max_prob = float(np.max(p_scene))

        p_cond = softmax_slice(11, 13)
        prob_good_cond = float(p_cond[1])

        p_green = softmax_slice(13, 16)
        prob_greenery = float(p_green[2])
        is_greenery_img = 1 if int(np.argmax(p_green)) == 2 else 0

        p_fin = softmax_slice(16, 19)
        prob_modern = float(p_fin[2])
        is_modern_img = 1 if int(np.argmax(p_fin)) == 2 else 0

        rec = meta_records[idx]
        rec.update({
            'auth_class': auth_type,
            'prob_real_photo': round(float(p_auth[0]), 4),
            'prob_cgi_render': round(float(p_auth[1]), 4),
            'prob_floorplan': round(float(p_auth[2]), 4),
            'prob_promo_flyer': round(float(p_auth[3]), 4),
            'auth_entropy': round(auth_entropy, 4),
            'scene_class': scene_type,
            'scene_confidence': round(scene_max_prob, 4),
            'prob_good_condition': round(prob_good_cond, 4),
            'is_good_condition_img': 1 if prob_good_cond >= 0.5 else 0,
            'prob_greenery': round(prob_greenery, 4),
            'is_greenery_img': is_greenery_img,
            'prob_modern_finishes': round(prob_modern, 4),
            'is_modern_finishes_img': is_modern_img
        })
        results.append(rec)

    return results

# %%
# 6. Smart Recursive Folder Discovery & Resumable Batch Execution
valid_exts = ('.jpg', '.jpeg', '.png', '.webp')
all_folders = []

if os.path.exists(IMAGES_DIR):
    for root, dirs, files in os.walk(IMAGES_DIR):
        if any(f.lower().endswith(valid_exts) for f in files):
            all_folders.append(root)
all_folders = sorted(all_folders)

if not all_folders:
    raise RuntimeError(f'[CRITICAL] No folders containing images were found under {IMAGES_DIR}. Verify the path.')

processed_ids = set()
if os.path.exists(IMAGE_LEVEL_CSV):
    try:
        existing_df = pd.read_csv(IMAGE_LEVEL_CSV, usecols=['listing_id'])
        processed_ids = set(existing_df['listing_id'].dropna().astype(str).unique())
        print(f'[RESUME] Found existing checkpoint in /DATA with {len(processed_ids)} processed folders.')
    except Exception as e:
        print(f'[INFO] Starting fresh checkpoint ({e}).')

remaining_folders = [f for f in all_folders if os.path.basename(f) not in processed_ids]
print(f'[INFO] Total image-bearing folders discovered: {len(all_folders)} | Remaining to process: {len(remaining_folders)}')

batch_buffer = []
folders_in_batch = 0

for folder_path in tqdm(remaining_folders, desc='Processing Listing Folders'):
    lid = os.path.basename(folder_path)
    img_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)])
    if not img_files:
        continue

    folder_res = analyze_folder_batch(folder_path, lid, img_files)
    if folder_res:
        batch_buffer.extend(folder_res)

    folders_in_batch += 1
    if folders_in_batch >= BATCH_SIZE and batch_buffer:
        df_batch = pd.DataFrame(batch_buffer)
        write_header = not os.path.exists(IMAGE_LEVEL_CSV)
        df_batch.to_csv(IMAGE_LEVEL_CSV, mode='a', header=write_header, index=False)
        batch_buffer = []
        folders_in_batch = 0

if batch_buffer:
    df_batch = pd.DataFrame(batch_buffer)
    write_header = not os.path.exists(IMAGE_LEVEL_CSV)
    df_batch.to_csv(IMAGE_LEVEL_CSV, mode='a', header=write_header, index=False)

if not os.path.exists(IMAGE_LEVEL_CSV):
    raise RuntimeError(f'[ERROR] {IMAGE_LEVEL_CSV} was not created because no readable images were processed.')

print('\n[SUCCESS] Image-level dataset saved to:', IMAGE_LEVEL_CSV)

# %%
# 7. Contamination-Free Listing Aggregation, Structural Merge & Statistical Diagnostics
df_img = pd.read_csv(IMAGE_LEVEL_CSV)
df_img['listing_id'] = df_img['listing_id'].astype(str)
df_img.drop_duplicates(subset=['listing_id', 'image_file'], keep='last', inplace=True)

df_img['is_valid_view'] = df_img['auth_class'].isin(['real_photo', 'cgi_render']).astype(int)
df_img['is_interior'] = (df_img['scene_class'].str.startswith('interior_') & (df_img['is_valid_view'] == 1)).astype(int)
df_img['is_exterior'] = (df_img['scene_class'].isin(['exterior_facade', 'outdoor_garden_patio', 'street_parking_gate']) & (df_img['is_valid_view'] == 1)).astype(int)

df_img['cond_valid'] = np.where(df_img['is_valid_view'] == 1, df_img['prob_good_condition'], np.nan)
df_img['green_valid'] = np.where(df_img['is_valid_view'] == 1, df_img['prob_greenery'], np.nan)
df_img['modern_valid_all'] = np.where(df_img['is_valid_view'] == 1, df_img['prob_modern_finishes'], np.nan)
df_img['modern_valid_int'] = np.where(df_img['is_interior'] == 1, df_img['prob_modern_finishes'], np.nan)

def calc_scene_entropy(series):
    counts = series.value_counts()
    probs = counts / counts.sum()
    return round(float(-np.sum(probs * np.log(probs))), 4)

agg = df_img.groupby('listing_id')

df_hedonic = pd.DataFrame({
    'n_photos_analyzed': agg['image_file'].count(),
    'n_valid_views': agg['is_valid_view'].sum(),
    'n_real_photos': agg['auth_class'].apply(lambda s: (s == 'real_photo').sum()),
    'n_cgi_renders': agg['auth_class'].apply(lambda s: (s == 'cgi_render').sum()),
    'n_floorplans': agg['auth_class'].apply(lambda s: (s == 'floorplan_map').sum()),
    'n_promo_flyers': agg['auth_class'].apply(lambda s: (s == 'promo_flyer').sum()),
    'cv_has_cgi_render': agg['auth_class'].apply(lambda s: int((s == 'cgi_render').any())),
    'cv_share_cgi_render': agg['auth_class'].apply(lambda s: round((s == 'cgi_render').mean(), 4)),
    'cv_has_nonphoto_noise': agg['auth_class'].apply(lambda s: int(s.isin(['floorplan_map', 'promo_flyer']).any())),
    'cv_has_exterior_facade': agg['scene_class'].apply(lambda s: int((s == 'exterior_facade').any())),
    'cv_has_kitchen_photo': agg['scene_class'].apply(lambda s: int((s == 'interior_kitchen').any())),
    'cv_has_bathroom_photo': agg['scene_class'].apply(lambda s: int((s == 'interior_bathroom').any())),
    'cv_has_interior_photo': agg['is_interior'].apply(lambda s: int(s.sum() > 0)),
    'cv_share_interior': agg['is_interior'].mean().round(4),
    'cv_share_exterior': agg['is_exterior'].mean().round(4),
    'cv_scene_diversity_entropy': agg['scene_class'].apply(calc_scene_entropy),
    'cv_n_unique_scenes': agg['scene_class'].nunique(),
    'cv_prob_good_condition_mean': agg['cond_valid'].mean().round(4),
    'cv_prob_greenery_max': agg['green_valid'].max().round(4),
    'cv_prob_modern_finishes_max': agg['modern_valid_all'].max().round(4),
    'cv_prob_modern_interior_only': agg['modern_valid_int'].max().round(4),
    'cv_brightness_mean': agg['brightness'].mean().round(2),
    'cv_contrast_mean': agg['contrast'].mean().round(2),
    'cv_sharpness_mean': agg['sharpness'].mean().round(2),
    'cv_colorfulness_mean': agg['colorfulness'].mean().round(2)
}).reset_index()

for col, fallback_col in [
    ('cv_prob_good_condition_mean', 'prob_good_condition'),
    ('cv_prob_greenery_max', 'prob_greenery'),
    ('cv_prob_modern_finishes_max', 'prob_modern_finishes')
]:
    fallback_vals = agg[fallback_col].mean().values
    df_hedonic[col] = df_hedonic[col].fillna(pd.Series(fallback_vals, index=df_hedonic.index)).round(4)

df_hedonic['cv_prob_modern_interior_only'] = df_hedonic['cv_prob_modern_interior_only'].fillna(df_hedonic['cv_prob_modern_finishes_max']).round(4)
df_hedonic['cv_good_condition'] = (df_hedonic['cv_prob_good_condition_mean'] >= 0.5).astype(int)
df_hedonic['cv_has_greenery'] = (df_hedonic['cv_prob_greenery_max'] >= 0.5).astype(int)
df_hedonic['cv_modern_finishes'] = (df_hedonic['cv_prob_modern_finishes_max'] >= 0.5).astype(int)

df_hedonic.to_csv(HEDONIC_CSV, index=False)
print(f'[SAVED] Hedonic listing-level CSV ({len(df_hedonic)} listings) -> {HEDONIC_CSV}')

# %%
# 8. Merge with Structural Dataset in /DATA & Build Self-Contained log.tex
METRO_MAPPING = {
    "Aguascalientes": "Aguascalientes", "Jesus Maria": "Aguascalientes", "Jesús María": "Aguascalientes", "San Francisco de los Romo": "Aguascalientes",
    "Leon": "Leon", "León": "Leon", "Purisima del Rincon": "Leon", "Purísima del Rincón": "Leon", "San Francisco del Rincon": "Leon", "San Francisco del Rincón": "Leon",
    "Zacatecas": "Zacatecas-Guadalupe", "Guadalupe": "Zacatecas-Guadalupe", "Morelos": "Zacatecas-Guadalupe", "Vetagrande": "Zacatecas-Guadalupe", "Trancoso": "Zacatecas-Guadalupe",
    "Queretaro": "Queretaro", "Querétaro": "Queretaro", "Corregidora": "Queretaro", "El Marques": "Queretaro", "El Marqués": "Queretaro"
}

has_structural = False
active_struct_path = STRUCTURAL_CSV if os.path.exists(STRUCTURAL_CSV) else (SPATIAL_CSV if os.path.exists(SPATIAL_CSV) else None)

if active_struct_path:
    df_struct = pd.read_csv(active_struct_path)
    df_struct['listing_id'] = df_struct['listing_id'].astype(str)
    if 'metro_area' not in df_struct.columns:
        if 'NOM_MET' in df_struct.columns:
            df_struct['metro_area'] = df_struct['NOM_MET'].fillna('Unmapped')
        elif 'municipality_reported' in df_struct.columns:
            df_struct['metro_area'] = df_struct['municipality_reported'].map(METRO_MAPPING).fillna('Unmapped')
        else:
            df_struct['metro_area'] = 'Unmapped'
    df_struct['price'] = pd.to_numeric(df_struct.get('price'), errors='coerce')
    df_struct['built_area_m2'] = pd.to_numeric(df_struct.get('built_area_m2'), errors='coerce')
    df_struct['price_m2'] = df_struct['price'] / df_struct['built_area_m2']

    df_merged = pd.merge(df_hedonic, df_struct, on='listing_id', how='inner')
    if len(df_merged) > 10:
        has_structural = True
        df_merged.to_csv(MERGED_AUDIT_CSV, index=False)
        print(f'[SAVED] Merged Structural + CV Audit CSV ({len(df_merged)} matched listings) -> {MERGED_AUDIT_CSV}')

sns.set_theme(style='whitegrid')

plt.figure(figsize=(11, 6))
sns.boxplot(data=df_img, x='scene_class', y='sharpness', hue='auth_class', showfliers=False)
plt.xticks(rotation=25, ha='right')
plt.title('Image-Level Laplacian Sharpness Across Scene Categories and Media Authenticity')
plt.xlabel('CLIP Scene Category')
plt.ylabel('Laplacian Sharpness Variance')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig01_scene_sharpness_boxplot.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_hedonic, x='cv_prob_good_condition_mean', hue='cv_has_cgi_render', common_norm=False, fill=True, alpha=0.4)
plt.title('Kernel Density of Structural Conservation Probability by CGI Render Presence')
plt.xlabel('Mean CLIP Probability of Good Structural Conservation')
plt.ylabel('Density')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig02_conservation_kde_by_cgi.png'), dpi=300)
plt.close()

cgi_grp = df_hedonic[df_hedonic['cv_has_cgi_render'] == 1]
real_grp = df_hedonic[df_hedonic['cv_has_cgi_render'] == 0]

def run_mwu(s1, s2):
    s1_c, s2_c = s1.dropna(), s2.dropna()
    if len(s1_c) > 2 and len(s2_c) > 2:
        u, p = stats.mannwhitneyu(s1_c, s2_c, alternative='two-sided')
        return s1_c.median(), s1_c.quantile(0.75) - s1_c.quantile(0.25), s2_c.median(), s2_c.quantile(0.75) - s2_c.quantile(0.25), u, p
    return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

test_rows = []
for label, col in [
    ('Conservation Prob.', 'cv_prob_good_condition_mean'),
    ('Greenery Prob. (Max)', 'cv_prob_greenery_max'),
    ('Modern Finishes Prob.', 'cv_prob_modern_finishes_max'),
    ('Interior Finishes Only', 'cv_prob_modern_interior_only'),
    ('Scene Diversity Entropy', 'cv_scene_diversity_entropy'),
    ('Mean Sharpness', 'cv_sharpness_mean'),
    ('Mean Brightness', 'cv_brightness_mean')
]:
    m1, iqr1, m0, iqr0, u_stat, p_val = run_mwu(cgi_grp[col], real_grp[col])
    test_rows.append({
        'Metric': label, 'CGI_Med': m1, 'CGI_IQR': iqr1,
        'Real_Med': m0, 'Real_IQR': iqr0, 'MWU_Stat': u_stat, 'p_value': p_val
    })

if has_structural:
    for label, col in [('Asking Price (MXN)', 'price'), ('Price per m2 (MXN)', 'price_m2'), ('Built Area (m2)', 'built_area_m2')]:
        m1, iqr1, m0, iqr0, u_stat, p_val = run_mwu(
            df_merged[df_merged['cv_has_cgi_render'] == 1][col],
            df_merged[df_merged['cv_has_cgi_render'] == 0][col]
        )
        test_rows.append({
            'Metric': f'{label} [CGI vs Real]', 'CGI_Med': m1, 'CGI_IQR': iqr1,
            'Real_Med': m0, 'Real_IQR': iqr0, 'MWU_Stat': u_stat, 'p_value': p_val
        })
    m1, iqr1, m0, iqr0, u_stat, p_val = run_mwu(
        df_merged[df_merged['cv_has_interior_photo'] == 1]['price_m2'],
        df_merged[df_merged['cv_has_interior_photo'] == 0]['price_m2']
    )
    test_rows.append({
        'Metric': 'Price/m2 [Interior vs Facade-Only]', 'CGI_Med': m1, 'CGI_IQR': iqr1,
        'Real_Med': m0, 'Real_IQR': iqr0, 'MWU_Stat': u_stat, 'p_value': p_val
    })

df_tests = pd.DataFrame(test_rows)

ct_scene_auth = pd.crosstab(df_img['scene_class'], df_img['auth_class'], margins=True)

scene_photometrics = df_img.groupby('scene_class').agg(
    Count=('image_file', 'count'),
    Share_Pct=('image_file', lambda s: round(len(s) / len(df_img) * 100, 2)),
    Scene_Conf_Med=('scene_confidence', 'median'),
    Bright_Med=('brightness', 'median'),
    Sharp_Med=('sharpness', 'median'),
    Sharp_IQR=('sharpness', lambda s: round(s.quantile(0.75) - s.quantile(0.25), 1)),
    Cond_Prob_Med=('prob_good_condition', 'median'),
    Modern_Prob_Med=('prob_modern_finishes', 'median')
).reset_index()

summary_cols = [
    'n_photos_analyzed', 'n_valid_views', 'cv_has_cgi_render', 'cv_share_cgi_render',
    'cv_has_nonphoto_noise', 'cv_has_exterior_facade', 'cv_has_interior_photo',
    'cv_has_kitchen_photo', 'cv_has_bathroom_photo', 'cv_share_interior',
    'cv_scene_diversity_entropy', 'cv_good_condition', 'cv_prob_good_condition_mean',
    'cv_has_greenery', 'cv_prob_greenery_max', 'cv_modern_finishes',
    'cv_prob_modern_finishes_max', 'cv_prob_modern_interior_only',
    'cv_brightness_mean', 'cv_contrast_mean', 'cv_sharpness_mean', 'cv_colorfulness_mean'
]
desc_stats = df_hedonic[summary_cols].describe(percentiles=[0.10, 0.25, 0.5, 0.75, 0.90]).T[
    ['count', 'mean', 'std', '10%', '25%', '50%', '75%', '90%']
].round(3)

corr_cols = [
    'cv_prob_good_condition_mean', 'cv_prob_greenery_max', 'cv_prob_modern_finishes_max',
    'cv_prob_modern_interior_only', 'cv_has_cgi_render', 'cv_share_interior',
    'cv_scene_diversity_entropy', 'cv_brightness_mean', 'cv_sharpness_mean'
]
if has_structural:
    corr_df_source = df_merged
    extra_cols = [c for c in ['price_m2', 'built_area_m2', 'dist_to_historic_center_km', 'is_planned_gated_enclave'] if c in df_merged.columns]
    corr_cols = extra_cols + corr_cols
else:
    corr_df_source = df_hedonic

spearman_corr = corr_df_source[corr_cols].corr(method='spearman').round(3)

metro_cv_table_tex = ""
if has_structural:
    metro_agg = df_merged.groupby('metro_area').agg(
        N=('listing_id', 'count'),
        Price_m2_Med=('price_m2', 'median'),
        CGI_Pct=('cv_has_cgi_render', lambda s: round(s.mean()*100, 1)),
        Int_Discl_Pct=('cv_has_interior_photo', lambda s: round(s.mean()*100, 1)),
        Entropy_Med=('cv_scene_diversity_entropy', 'median'),
        Cond_Med=('cv_prob_good_condition_mean', 'median'),
        Green_Med=('cv_prob_greenery_max', 'median'),
        Modern_Int_Med=('cv_prob_modern_interior_only', 'median'),
        Sharp_Med=('cv_sharpness_mean', 'median')
    ).reset_index().sort_values(by='N', ascending=False)
    metro_cv_table_tex = metro_agg.to_latex(index=False, float_format='%.3f', caption='Metropolitan Breakdown of Visual Hedonic Attributes and Unit Prices', label='tab:metro_cv_breakdown')

with open(TABLE_SCENE_TEX, 'w', encoding='utf-8') as f:
    f.write(ct_scene_auth.to_latex(caption='Cross-Tabulation of Scene Taxonomy and Media Authenticity', label='tab:cv_scene_auth'))

with open(TABLE_STATS_TEX, 'w', encoding='utf-8') as f:
    f.write(desc_stats.to_latex(caption='Decile and Quartile Distribution of Listing-Level Computer Vision Variables', label='tab:cv_summary_stats', float_format='%.3f'))

with open(TABLE_TESTS_TEX, 'w', encoding='utf-8') as f:
    f.write(df_tests.to_latex(index=False, caption='Mann-Whitney U Hypothesis Tests for CGI Renders and Interior Disclosure', label='tab:cv_mwu_tests', float_format='%.4f'))

n_listings = len(df_hedonic)
n_images = len(df_img)
cgi_count = int(df_hedonic['cv_has_cgi_render'].sum())
cgi_pct = df_hedonic['cv_has_cgi_render'].mean() * 100.0
noise_count = int(df_hedonic['cv_has_nonphoto_noise'].sum())
noise_pct = df_hedonic['cv_has_nonphoto_noise'].mean() * 100.0
int_discl_pct = df_hedonic['cv_has_interior_photo'].mean() * 100.0
mod_diff = (df_hedonic['cv_prob_modern_interior_only'] - df_hedonic['cv_prob_modern_finishes_max']).abs().mean()

log_tex_content = f"""% ==============================================================================
% MASTER COMPUTER VISION & HEDONIC DIAGNOSTIC AUDIT (log.tex)
% Generated at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M-%S')}
% ==============================================================================
\\documentclass[11pt,a4paper]{{article}}
\\usepackage[utf8]{{inputenc}}
\\usepackage{{booktabs}}
\\usepackage{{geometry}}
\\usepackage{{float}}
\\usepackage{{graphicx}}
\\geometry{{margin=0.85in}}
\\begin{{document}}

\\section*{{1. Corpus Architecture and Visual Authenticity Audit}}
The computer vision pipeline processed {n_images} individual image files distributed across {n_listings} unique single-family housing listings, averaging {df_hedonic['n_photos_analyzed'].mean():.2f} photographs per property. Evaluating image-level media authenticity via zero-shot contrastive language-image pretraining reveals that real camera photographs account for {(df_img['auth_class']=='real_photo').mean()*100:.2f} percent of the visual corpus, whereas three-dimensional architectural renders represent {(df_img['auth_class']=='cgi_render').mean()*100:.2f} percent, two-dimensional blueprints represent {(df_img['auth_class']=='floorplan_map').mean()*100:.2f} percent, and promotional banners represent {(df_img['auth_class']=='promo_flyer').mean()*100:.2f} percent. At the property level, {cgi_count} listings ({cgi_pct:.2f} percent) incorporate at least one digital render indicative of presale or new-build marketing, while {noise_count} listings ({noise_pct:.2f} percent) contain non-photographic blueprints or text flyers that were masked prior to computing hedonic quality scores.

\\section*{{2. Strategic Room Disclosure and Scene Taxonomy}}
Real estate brokers selectively curate visual galleries to maximize buyer appeal, creating potential information asymmetries when interior rooms are withheld. Across the analyzed listings, {int_discl_pct:.2f} percent of properties disclose at least one valid interior room photograph, while {100.0 - int_discl_pct:.2f} percent restrict visual disclosure exclusively to exterior facades, street views, or gated entrances. Measuring gallery composition through Shannon scene diversity entropy yields a median entropy of {df_hedonic['cv_scene_diversity_entropy'].median():.3f} (interquartile range {df_hedonic['cv_scene_diversity_entropy'].quantile(0.75)-df_hedonic['cv_scene_diversity_entropy'].quantile(0.25):.3f}). Furthermore, restricting modern finish scoring to validated interior rooms shifts the mean finish probability by {mod_diff:.4f} relative to unmasked aggregation, preventing exterior facades from diluting interior renovation signals.

{ct_scene_auth.to_latex(caption='Image-Level Cross-Tabulation: Scene Taxonomy versus Media Authenticity', label='tab:log_scene_auth')}

\\section*{{3. Numerical Summary of Boxplot Distributions by Scene Class}}
To permit full diagnostic inspection of the visual distributions generated in Figure 1 without inspecting raster files, Table 2 reports the exact median and interquartile range of Laplacian sharpness, luminance, classifier confidence, and hedonic probabilities across each of the seven scene categories.

\\begin{{table}}[H]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{%
{scene_photometrics.to_latex(index=False, float_format='%.2f')}
}}
\\caption{{Exact Quantile and Photometric Summary Underlying Scene-Level Boxplots}}
\\end{{table}}

\\section*{{4. Listing-Level Decile and Quartile Distributions}}
Table 3 reports the parametric and non-parametric distributional moments (mean, standard deviation, 10th, 25th, 50th, 75th, and 90th percentiles) for all twenty-two listing-level visual and photometric variables exported to the master hedonic dataset.

\\begin{{table}}[H]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{%
{desc_stats.to_latex(float_format='%.3f')}
}}
\\caption{{Listing-Level Distributional Moments of Computer Vision and Photometric Indicators}}
\\end{{table}}

\\section*{{5. Non-Parametric Hypothesis Tests: CGI Renders and Disclosure Regimes}}
Table 4 evaluates whether listings marketed with architectural CGI renders differ systematically in perceived structural conservation, greenery, finish quality, photometric sharpness, and asking prices compared to listings documented exclusively with real photographs.

\\begin{{table}}[H]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{%
{df_tests.to_latex(index=False, float_format='%.4f')}
}}
\\caption{{Mann-Whitney U Tests Comparing CGI Render Regimes and Interior Disclosure Regimes}}
\\end{{table}}

\\section*{{6. Metropolitan Submarket Breakdown and Spearman Rank Correlations}}
{metro_cv_table_tex if has_structural else 'Structural CSV was not found in /DATA; verify property\\_listings\\_multicity\\_filtered\\_15km.csv is located in /DATA.'}

\\begin{{table}}[H]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{%
{spearman_corr.to_latex(float_format='%.3f')}
}}
\\caption{{Spearman Rank Correlation Matrix Across Structural and Computer Vision Features}}
\\end{{table}}

\\end{{document}}
"""

with open(LOG_TEX, 'w', encoding='utf-8') as f:
    f.write(log_tex_content)

print(f'[SAVED] All CSVs, LaTeX tables, and master log.tex saved in: {DATA_DIR}')

Mounted at /content/drive
[INFO] Running CLIP inference on device: cuda
[INFO] Images directory exists: True
[INFO] 15km Structural CSV exists: True


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

[RESUME] Found existing checkpoint in /DATA with 3150 processed folders.
[INFO] Total image-bearing folders discovered: 3150 | Remaining to process: 0


Processing Listing Folders: 0it [00:00, ?it/s]


[SUCCESS] Image-level dataset saved to: /content/drive/My Drive/ZAC_GPE_PRICES/DATA/cv_image_level_details.csv
[SAVED] Hedonic listing-level CSV (3150 listings) -> /content/drive/My Drive/ZAC_GPE_PRICES/DATA/cv_features_hedonic.csv
[SAVED] All CSVs, LaTeX tables, and master log.tex saved in: /content/drive/My Drive/ZAC_GPE_PRICES/DATA
